# PhysAR-IDS -- NB12 (v6): federated learning across PLC clients -- privacy, heterogeneity, poisoning and adversarial robustness

**What the v4 version could and could not say.** The v4 chapter compared federated MLPs (F1 0.6141) with a *centralised XGBoost* (0.7225) and called the difference "the privacy cost". That mixes two effects (model family and federation). It also reported Trimmed Mean "robustness" without a FedAvg-under-attack baseline, a PhysFed result without a control for what the physics weighting actually changes, and a single DP point. This notebook rebuilds the experiment on the v6 pipeline with those gaps closed, 5 seeds [42, 7, 123, 2024, 99], mean +/- std, paired tests with Holm, and burst-cluster bootstrap intervals.

**Setup.** Nine clients; the shared model is an MLP 93-64-32-6 (the NB3 surrogate architecture); 30 rounds; 5 local epochs per round (Adam, lr 1e-3, batch 1024, optimiser re-initialised each round); FedAvg weights = sample counts. **Partitions are made at the burst level** (rows sharing class, PLC and timestamp are identical copies and are never split across clients): `iid` (stratified random bursts), `dirichlet` label skew with alpha 0.5 (mild) and 0.1 (severe), and `plc` (clients = real PLCs, mapped onto nine clients). The global model after the **final round** is evaluated once on the temporal test split (no model selection on test). All configurations of a seed start from the same initial weights.

**Configurations.**
* **References:** `central_150` (pooled data, 150 epochs = the same number of passes over the data as 30 rounds x 5 local epochs; the *declared* reference), `central_30` (sensitivity), `local_only` (each client alone for 150 epochs, mean over clients). The XGBoost F1 (0.8635, NB2) is quoted as context only: the privacy cost is FedAvg versus `central_150`, the same MLP.
* **Federation:** `fedavg_iid`, `fedavg_a05`, `fedprox_a05` (mu 0.01), `fedavg_a01`, `fedprox_a01` (tests the v4 claim that FedProx would separate from FedAvg under severe heterogeneity), `fedavg_plc`.
* **PhysFed and its control:** `physfed_a05` weights clients by their local PCR (share of gradient x input attribution on the 33 physical-channel features; a cheap proxy for the SHAP-based PCR, computed on the client's own rows). `uniform_a05` (equal client weights) is the control: if PCR values are similar across clients, PhysFed is just uniform weighting.
* **Poisoning:** one malicious client (index 0) returns -10 x its honest update every round. `fedavg_byz_a05`, `trimmed_byz_a05` (coordinate-wise trimmed mean, trim 0.2 = one client per end), and the clean `trimmed_a05`.
* **Differential privacy:** `dp_s*` -- clip each client update to L2 norm C = 1, average, add Gaussian noise (sigma x C)/9 per coordinate. The (epsilon, delta = 1e-5) guarantee is **client-level**, from Renyi-DP composition over 30 full-participation rounds (add/remove adjacency), printed next to each sigma. A sigma sweep replaces the single v4 point.
* **Adversarial robustness:** white-box PGD (epsilon 0.1 sigma, 10 steps) against each final global model on the test attack rows. Evasion = share of attack rows classified benign after the attack, **including those already missed on clean data** (the clean value is printed beside it).

**Limits to carry into the text.** Federation is simulated on one machine (no hardware, no communication faults). Sending weights instead of data is not a privacy guarantee on its own (updates can leak); only the DP rows carry a formal bound. The poisoning model is one scaled sign-flip attacker, not an adaptive one. PCR here is a gradient x input proxy for NB8's TreeSHAP PCR.

**Needs:** `phase1_data_v6.pkl`. **Runtime:** about 1.5-2.5 h on the local GPU; progress is saved after every configuration, so an interrupted run resumes.

## 0. Setup

In [ ]:
import os, sys, random, time, json, warnings, math
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.metrics import roc_auc_score
import matplotlib; import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False
N_BOOT = 2000
PGD_EPS = 0.1; PGD_ITERS = 10
N_CLIENTS = 9; ROUNDS = 30; LOCAL_EPOCHS = 5; BATCH = 1024; LR = 1e-3
TRIM = 0.2; BYZ_SCALE = 10.0; DP_CLIP = 1.0; DP_DELTA = 1e-5; DP_SIGMAS = [0.1, 1.0, 3.0]
PCR_N = 2000; MIN_BURSTS = 20

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} backend: {"cpu(hist)" if _cpu else "gpu(cuda,hist)"}')
print('Working directory:', os.getcwd())

## 1. Load v6 data, bursts, partitions

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']).astype(np.int64); y_test = np.asarray(data['y_test']).astype(np.int64)
plc_tr, plc_te = np.asarray(data['PLCID_train']), np.asarray(data['PLCID_test']); ts_tr, ts_te = np.asarray(data['timestamp_train']), np.asarray(data['timestamp_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
IS_CYBER = np.zeros(N_FEAT, bool); IS_CYBER[CYBER_IDX] = True; PHYS_IDX = np.where(~IS_CYBER)[0]; assert len(PHYS_IDX) == 33 or SMOKE_TEST
def burst_ids(y, plc, ts): return pd.DataFrame({'y': y, 'p': plc, 't': ts}).groupby(['y', 'p', 't'], sort=False).ngroup().to_numpy()
bid_tr, bid_te = burst_ids(y_train, plc_tr, ts_tr), burst_ids(y_test, plc_te, ts_te)
order = np.argsort(bid_tr, kind='stable'); starts = np.r_[0, np.flatnonzero(np.diff(bid_tr[order])) + 1]; ends = np.r_[starts[1:], len(order)]
b_rows = [order[s:e] for s, e in zip(starts, ends)]; b_cls = np.array([y_train[r[0]] for r in b_rows]); b_plc = np.array([plc_tr[r[0]] for r in b_rows])
ia = np.flatnonzero(y_test != 0)
print(f'train {len(y_train):,} rows in {len(b_rows)} bursts | test {len(y_test):,} rows in {len(np.unique(bid_te))} bursts | PLCs in train: {len(np.unique(plc_tr))} | bursts per class {np.bincount(b_cls, minlength=6).tolist()}')
X_dev = torch.from_numpy(X_train).to(DEVICE); y_dev = torch.from_numpy(y_train).to(DEVICE); Xt_dev = torch.from_numpy(X_test).to(DEVICE)

def part_iid(rng):
    cl = [[] for _ in range(N_CLIENTS)]
    for c in range(6):
        for j, b in enumerate(rng.permutation(np.where(b_cls == c)[0])): cl[j % N_CLIENTS].append(b)
    return cl
def part_dirichlet(rng, alpha):
    for _ in range(200):
        cl = [[] for _ in range(N_CLIENTS)]
        for c in range(6):
            bs = rng.permutation(np.where(b_cls == c)[0]); p = rng.dirichlet(alpha * np.ones(N_CLIENTS)); cuts = (np.cumsum(p) * len(bs)).astype(int)[:-1]
            for k, chunk in enumerate(np.split(bs, cuts)): cl[k].extend(chunk.tolist())
        if min(len(x) for x in cl) >= MIN_BURSTS: return cl
    raise RuntimeError('no Dirichlet partition with every client above MIN_BURSTS')
def part_plc(rng):
    u = np.unique(b_plc); m = {p: i % N_CLIENTS for i, p in enumerate(u)}; cl = [[] for _ in range(N_CLIENTS)]
    for b in range(len(b_rows)): cl[m[b_plc[b]]].append(b)
    return cl
def make_partition(kind, seed):
    rng = np.random.RandomState(seed + 31)
    cl = {'iid': lambda: part_iid(rng), 'a05': lambda: part_dirichlet(rng, 0.5), 'a01': lambda: part_dirichlet(rng, 0.1), 'plc': lambda: part_plc(rng)}[kind]()
    cl = [x for x in cl if len(x) > 0]
    return [torch.from_numpy(np.concatenate([b_rows[b] for b in x])).to(DEVICE) for x in cl]
rows = []
for kind in ('iid', 'a05', 'a01', 'plc'):
    pt = make_partition(kind, SEEDS[0])
    for k, idx in enumerate(pt): rows.append({'partition': kind, 'client': k, 'rows': len(idx), **{c: int((y_dev[idx] == i).sum()) for i, c in enumerate(class_names)}})
PART_TAB = pd.DataFrame(rows); print(PART_TAB.to_string(index=False))

## 2. Model, local training, aggregation rules, evaluation

In [ ]:
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_features, hidden), nn.ReLU(), nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, n_classes))
    def forward(self, x): return self.net(x)
def pgd(model, X, y, eps=PGD_EPS, n_iter=PGD_ITERS, bs=65536):
    model.eval(); alpha = eps / 4; out = np.empty_like(X)
    for s in range(0, len(X), bs):
        yt = torch.from_numpy(y[s:s + bs]).long().to(DEVICE); X0 = torch.from_numpy(X[s:s + bs]).float().to(DEVICE); Xa = X0.clone()
        for _ in range(n_iter):
            Xa.requires_grad_(True); g = torch.autograd.grad(F.cross_entropy(model(Xa), yt), Xa)[0]; Xa = Xa.detach() + alpha * g.sign(); Xa = X0 + torch.clamp(Xa - X0, -eps, eps)
        out[s:s + bs] = Xa.detach().cpu().numpy()
    return out

In [ ]:
def conf_by_burst(y, pred, bid):
    u, inv = np.unique(bid, return_inverse=True); return np.bincount(inv * 36 + y * 6 + pred, minlength=len(u) * 36).reshape(len(u), 6, 6).astype(float)
def f1_from_conf(C):
    tp = np.einsum('...ii->...i', C); rs = C.sum(-1); cs = C.sum(-2)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1 = np.where(rs + cs > 0, 2 * tp / (rs + cs), 0.0); rec = np.where(rs > 0, tp / rs, np.nan)
    sup = rs > 0; macro = (f1 * sup).sum(-1) / sup.sum(-1); weighted = (f1 * rs).sum(-1) / rs.sum(-1)
    return macro, weighted, f1, rec

In [ ]:
def get_vec(m): return torch.nn.utils.parameters_to_vector(m.parameters()).detach().clone()
def set_vec(m, v): torch.nn.utils.vector_to_parameters(v.clone(), m.parameters())
N_PARAMS = int(get_vec(SurrogateMLP(N_FEAT)).numel())

def local_train(m, w0, idx, epochs, mu, gen):
    set_vec(m, w0); opt = torch.optim.Adam(m.parameters(), lr=LR); m.train()
    for _ in range(epochs):
        perm = idx[torch.randperm(len(idx), generator=gen, device=DEVICE)]
        for s in range(0, len(perm), BATCH):
            b = perm[s:s + BATCH]; loss = F.cross_entropy(m(X_dev[b]), y_dev[b])
            if mu > 0: loss = loss + 0.5 * mu * ((torch.nn.utils.parameters_to_vector(m.parameters()) - w0) ** 2).sum()
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    return get_vec(m)

@torch.no_grad()
def predict(m, X):
    m.eval(); return torch.cat([m(X[s:s + 65536]).argmax(1) for s in range(0, len(X), 65536)]).cpu().numpy()
def score_pred(pred):
    C = conf_by_burst(y_test, pred, bid_te); mac, wei, f1c, rec = f1_from_conf(C.sum(0)); return C.reshape(len(C), 36), float(wei), float(mac), rec
def pcr_of(m, idx, gen):
    sel = idx[torch.randperm(len(idx), generator=gen, device=DEVICE)[:PCR_N]]; x = X_dev[sel].clone().requires_grad_(True); m.eval()
    out = m(x); g = torch.autograd.grad(out.gather(1, out.argmax(1, keepdim=True)).sum(), x)[0]; a = (g * x).abs().mean(0); m.train()
    return float(a[torch.from_numpy(PHYS_IDX).to(DEVICE)].sum() / a.sum())
def dp_eps(sigma, T, delta):
    best = np.inf
    for a in np.concatenate([np.arange(1.01, 10, 0.01), np.arange(10, 512, 0.5)]): best = min(best, T * a / (2 * sigma ** 2) + np.log(1 / delta) / (a - 1))
    return float(best)
def attack_eval(m, pred_clean):
    Xa = pgd(m, X_test[ia], y_test[ia]); pa = predict(m, torch.from_numpy(Xa).to(DEVICE)); pred = pred_clean.copy(); pred[ia] = pa
    _, wei, mac, rec = score_pred(pred); return {'evasion_clean': float((pred_clean[ia] == 0).mean()), 'evasion_pgd': float((pa == 0).mean()), 'f1_pgd': wei, 'recall_pgd': rec}

def fl_run(seed, part, agg='fedavg', mu=0.0, byz=False, sigma=None):
    torch.manual_seed(seed); m = SurrogateMLP(N_FEAT).to(DEVICE); w = get_vec(m); gen = torch.Generator(device=DEVICE); gen.manual_seed(seed * 7 + 1)
    sizes = torch.tensor([float(len(i)) for i in part], device=DEVICE); curve = []; clip_frac = []; pcr_hist = []
    for rd in range(ROUNDS):
        deltas = []; pcrs = []
        for k, idx in enumerate(part):
            wk = local_train(m, w, idx, LOCAL_EPOCHS, mu, gen); d = wk - w
            if agg == 'physfed': set_vec(m, wk); pcrs.append(pcr_of(m, idx, gen))
            if byz and k == 0: d = -BYZ_SCALE * d
            deltas.append(d)
        D = torch.stack(deltas)
        if agg == 'fedavg': upd = ((sizes / sizes.sum())[:, None] * D).sum(0)
        elif agg == 'uniform': upd = D.mean(0)
        elif agg == 'physfed': pw = torch.tensor(pcrs, device=DEVICE); upd = ((pw / pw.sum())[:, None] * D).sum(0); pcr_hist.append(pcrs)
        elif agg == 'trimmed': t = int(TRIM * len(D)); upd = D.sort(0)[0][t:len(D) - t].mean(0)
        elif agg == 'dp':
            nr = D.norm(dim=1); clip_frac.append(float((nr > DP_CLIP).float().mean())); Dc = D * torch.clamp(DP_CLIP / nr, max=1.0)[:, None]
            upd = (Dc.sum(0) + sigma * DP_CLIP * torch.randn(N_PARAMS, device=DEVICE, generator=gen)) / len(D)
        else: raise ValueError(agg)
        w = w + upd; set_vec(m, w); curve.append(score_pred(predict(m, Xt_dev))[1])
    set_vec(m, w); pred = predict(m, Xt_dev); C, wei, mac, rec = score_pred(pred)
    out = {'C': C, 'weighted': wei, 'macro': mac, 'recall_pc': rec, 'curve': np.array(curve), 'attack': attack_eval(m, pred), 'n_clients': len(part), 'client_rows': [len(i) for i in part]}
    if clip_frac: out['clip_frac'] = float(np.mean(clip_frac)); out['eps'] = dp_eps(sigma, ROUNDS, DP_DELTA)
    if pcr_hist: out['pcr_mean'] = float(np.mean(pcr_hist)); out['pcr_client_std'] = float(np.mean([np.std(p) for p in pcr_hist])); out['pcr_weight_spread'] = float(np.mean([max(p) / sum(p) * len(p) for p in pcr_hist]))
    return out
def central_run(seed, epochs):
    torch.manual_seed(seed); m = SurrogateMLP(N_FEAT).to(DEVICE); w = get_vec(m); gen = torch.Generator(device=DEVICE); gen.manual_seed(seed * 7 + 1)
    w = local_train(m, w, torch.arange(len(y_train), device=DEVICE), epochs, 0.0, gen); set_vec(m, w); pred = predict(m, Xt_dev); C, wei, mac, rec = score_pred(pred)
    return {'C': C, 'weighted': wei, 'macro': mac, 'recall_pc': rec, 'attack': attack_eval(m, pred)}
def local_only_run(seed, part):
    torch.manual_seed(seed); m0 = SurrogateMLP(N_FEAT).to(DEVICE); w0 = get_vec(m0); gen = torch.Generator(device=DEVICE); gen.manual_seed(seed * 7 + 1); res = []
    for idx in part:
        m = SurrogateMLP(N_FEAT).to(DEVICE); w = local_train(m, w0, idx, ROUNDS * LOCAL_EPOCHS, 0.0, gen); set_vec(m, w); pred = predict(m, Xt_dev); C, wei, mac, rec = score_pred(pred); res.append((C, wei, mac, rec, attack_eval(m, pred)))
    return {'C': np.mean([r[0] for r in res], 0), 'weighted': float(np.mean([r[1] for r in res])), 'macro': float(np.mean([r[2] for r in res])), 'recall_pc': np.nanmean([r[3] for r in res], 0), 'weighted_per_client': [r[1] for r in res],
            'attack': {k: (np.mean([r[4][k] for r in res], 0) if k == 'recall_pgd' else float(np.mean([r[4][k] for r in res]))) for k in res[0][4]}, 'n_clients': len(part)}
print(f'model parameters: {N_PARAMS:,} ({N_PARAMS * 4 / 1024:.1f} KiB per update)')

## 3. Run all configurations

Results are written after every configuration (file `fl_seed{s}_v6.pkl`); delete it to force a rerun.

In [ ]:
CONFIGS = {   # name: (kind, partition, kwargs)
    'central_150': ('central', None, dict(epochs=ROUNDS * LOCAL_EPOCHS)), 'central_30': ('central', None, dict(epochs=30)), 'local_only': ('local', 'a05', {}),
    'fedavg_iid': ('fl', 'iid', dict(agg='fedavg')), 'fedavg_a05': ('fl', 'a05', dict(agg='fedavg')), 'fedprox_a05': ('fl', 'a05', dict(agg='fedavg', mu=0.01)),
    'fedavg_a01': ('fl', 'a01', dict(agg='fedavg')), 'fedprox_a01': ('fl', 'a01', dict(agg='fedavg', mu=0.01)), 'fedavg_plc': ('fl', 'plc', dict(agg='fedavg')),
    'uniform_a05': ('fl', 'a05', dict(agg='uniform')), 'physfed_a05': ('fl', 'a05', dict(agg='physfed')),
    'trimmed_a05': ('fl', 'a05', dict(agg='trimmed')), 'fedavg_byz_a05': ('fl', 'a05', dict(agg='fedavg', byz=True)), 'trimmed_byz_a05': ('fl', 'a05', dict(agg='trimmed', byz=True)),
    **{f'dp_s{s_}': ('fl', 'a05', dict(agg='dp', sigma=s_)) for s_ in DP_SIGMAS}}
if SMOKE_TEST: ROUNDS = 3; LOCAL_EPOCHS = 1; CONFIGS['central_150']= ('central', None, dict(epochs=3)); CONFIGS['central_30']= ('central', None, dict(epochs=2))
RES = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'fl_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = json.dumps([seed, N_CLIENTS, ROUNDS, LOCAL_EPOCHS, BATCH, LR, TRIM, BYZ_SCALE, DP_CLIP, DP_SIGMAS, PGD_EPS, PGD_ITERS, float(X_train[:5].sum()), len(y_train), SMOKE_TEST, 'nb12-v1'], sort_keys=True)
    saved = joblib.load(pf) if os.path.exists(pf) else {}
    done = saved.get('res', {}) if saved.get('fp') == cfg_fp else {}
    print(f'=== seed {seed}: {len(done)}/{len(CONFIGS)} configurations already saved', flush=True); parts = {}
    for name, (kind, pk, kw) in CONFIGS.items():
        if name in done: continue
        t0 = time.time()
        if pk and pk not in parts: parts[pk] = make_partition(pk, seed)
        if kind == 'central': r = central_run(seed, **kw)
        elif kind == 'local': r = local_only_run(seed, parts[pk])
        else: r = fl_run(seed, parts[pk], **kw)
        done[name] = r; joblib.dump({'fp': cfg_fp, 'res': done}, pf)
        print(f'  {name:16s} F1 {r["weighted"]:.4f} (macro {r["macro"]:.4f}) benign recall {r["recall_pc"][0]:.3f} | PGD evasion {r["attack"]["evasion_pgd"]:.3f} (clean {r["attack"]["evasion_clean"]:.3f}) | {time.time()-t0:.0f}s' + (f' | eps {r["eps"]:.1f}, clipped {r["clip_frac"]:.2f}' if 'eps' in r else ''), flush=True)
    RES[seed] = done
print(f'all seeds done in {(time.time()-t_all)/60:.1f} min')

## 4. Statistics helpers

In [ ]:
def ptest(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    if len(d) < 2: return float('nan')
    return 1.0 if d.std(ddof=1) == 0 and d.mean() == 0 else (float('nan') if d.std(ddof=1) == 0 else float(stats.ttest_rel(a, b).pvalue))
def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0.0
    for rk, i in enumerate(o): run = max(run, (len(p) - rk) * p[i]); adj[i] = min(1.0, run)
    return adj
def dz(a, b):
    d = np.asarray(a, float) - np.asarray(b, float); s = d.std(ddof=1) if len(d) > 1 else 0
    return float(d.mean() / s) if s > 0 else float('nan')
def one_sample_p(x, mu):
    x = np.asarray(x, float)
    if len(x) < 2 or x.std(ddof=1) == 0: return float('nan')
    return float(stats.ttest_1samp(x, mu).pvalue)
def fp(p): return '< 0.0001' if (p == p and p < 5e-5) else (f'{p:.4f}' if p == p else 'n/a')
ms = lambda x, d=3: f'{np.mean(x):.{d}f} +/- {(np.std(x, ddof=1) if len(x) > 1 else 0):.{d}f}'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); T = {}

In [ ]:
G = lambda name, f: np.array([RES[s][name][f] for s in SEEDS])                   # per-seed scalar
GA = lambda name, f: np.array([RES[s][name]['attack'][f] for s in SEEDS])
def boot_ci(name, metric='weighted', B=N_BOOT):
    # burst-cluster bootstrap of the seed-averaged metric: same resampled bursts for every seed
    C = np.stack([RES[s][name]['C'] for s in SEEDS]); nb = C.shape[1]; W = np.random.RandomState(0).multinomial(nb, np.ones(nb) / nb, size=B).astype(float)
    vals = np.stack([f1_from_conf((W @ C[i]).reshape(-1, 6, 6))[1 if metric == 'weighted' else 0] for i in range(len(SEEDS))]).mean(0)
    return np.percentile(vals, [2.5, 97.5])
LABEL = {'central_150': 'centralised MLP, 150 epochs (reference)', 'central_30': 'centralised MLP, 30 epochs', 'local_only': 'local-only (mean over clients, alpha 0.5)', 'fedavg_iid': 'FedAvg IID', 'fedavg_a05': 'FedAvg Dirichlet 0.5', 'fedprox_a05': 'FedProx mu 0.01, Dirichlet 0.5',
         'fedavg_a01': 'FedAvg Dirichlet 0.1', 'fedprox_a01': 'FedProx mu 0.01, Dirichlet 0.1', 'fedavg_plc': 'FedAvg real-PLC clients', 'uniform_a05': 'uniform-weight FedAvg (control), 0.5', 'physfed_a05': 'PhysFed (PCR weights), 0.5',
         'trimmed_a05': 'Trimmed Mean (clean), 0.5', 'fedavg_byz_a05': 'FedAvg + 1 Byzantine, 0.5', 'trimmed_byz_a05': 'Trimmed Mean + 1 Byzantine, 0.5', **{f'dp_s{s_}': f'DP-FL sigma {s_}, 0.5' for s_ in DP_SIGMAS}}
T = {}; pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)

## 5. Results

In [ ]:
names = list(CONFIGS)
print('5a. Final-round test performance (mean +/- std over 5 seeds). burst CI = bootstrap over test bursts of the seed-averaged weighted F1. Context: centralised XGBoost (NB2) is 0.8635 +/- 0.0082 -- a different model family, not the privacy reference.')
rows = []
for n in names:
    ci = boot_ci(n); rows.append({'configuration': LABEL[n], 'weighted F1': ms(G(n, 'weighted'), 4), 'burst CI 95%': f'[{ci[0]:.3f}, {ci[1]:.3f}]', 'macro F1': ms(G(n, 'macro'), 4), 'benign recall': ms(np.array([RES[s][n]['recall_pc'][0] for s in SEEDS])),
                 'gap vs central_150': f'{(G(n, "weighted") - G("central_150", "weighted")).mean():+.4f}', 'dz': f'{dz(G(n, "weighted"), G("central_150", "weighted")):.2f}' if n != 'central_150' else ''})
T['main'] = pd.DataFrame(rows); print(T['main'].to_string(index=False))

print('\n5b. Per-class recall (mean +/- std over seeds)')
T['perclass'] = pd.DataFrame([{'configuration': LABEL[n], **{c: ms(np.array([RES[s][n]['recall_pc'][i] for s in SEEDS])) for i, c in enumerate(class_names)}} for n in names]); print(T['perclass'].to_string(index=False))

print('\n5c. Planned paired comparisons (paired over seeds; Holm over this table; dz = paired effect size; difference = first minus second)')
PAIRS = [('fedavg_iid', 'central_150', 'cost of federation, IID'), ('fedavg_a05', 'central_150', 'cost of federation, mild skew (the privacy-accuracy cost)'), ('fedavg_a01', 'central_150', 'cost of federation, severe skew'), ('fedavg_plc', 'central_150', 'cost of federation, real PLC clients'),
         ('fedavg_iid', 'fedavg_a05', 'IID vs mild skew'), ('fedavg_a05', 'fedavg_a01', 'mild vs severe skew'), ('fedprox_a05', 'fedavg_a05', 'FedProx vs FedAvg, mild skew'), ('fedprox_a01', 'fedavg_a01', 'FedProx vs FedAvg, severe skew'),
         ('fedavg_a05', 'local_only', 'federation vs local-only'), ('physfed_a05', 'fedavg_a05', 'PhysFed vs FedAvg'), ('physfed_a05', 'uniform_a05', 'PhysFed vs uniform weights (control)'), ('uniform_a05', 'fedavg_a05', 'uniform vs sample-count weights'),
         ('trimmed_a05', 'fedavg_a05', 'Trimmed Mean overhead (clean)'), ('fedavg_byz_a05', 'fedavg_a05', 'FedAvg: effect of 1 Byzantine client'), ('trimmed_byz_a05', 'trimmed_a05', 'Trimmed Mean: effect of 1 Byzantine client'), ('trimmed_byz_a05', 'fedavg_byz_a05', 'Trimmed Mean vs FedAvg under attack'),
         ('central_30', 'central_150', 'central 30 vs 150 epochs (sensitivity)')] + [(f'dp_s{s_}', 'fedavg_a05', f'DP sigma {s_} vs FedAvg') for s_ in DP_SIGMAS]
pl = [ptest(G(a, 'weighted'), G(b, 'weighted')) for a, b, _ in PAIRS]; ph = holm(pl)
T['compare'] = pd.DataFrame([{'comparison': lab, 'first': a, 'second': b, 'difference in F1': f'{(G(a, "weighted") - G(b, "weighted")).mean():+.4f}', 'dz': f'{dz(G(a, "weighted"), G(b, "weighted")):.2f}', 'p (Holm)': fp(ph[i])} for i, (a, b, lab) in enumerate(PAIRS)]); print(T['compare'].to_string(index=False))
print('   dz and p are only as informative as 5 seeds allow; the seeds vary initialisation, partition and training order, not the (fixed) temporal test split -- read the burst CIs in 5a for test-set uncertainty.')

In [ ]:
print('5d. Convergence: weighted F1 after selected rounds (mean over seeds)')
rds = [r_ for r_ in (1, 5, 10, 20, ROUNDS) if r_ <= ROUNDS]; rows = []
for n in names:
    if 'curve' in RES[SEEDS[0]][n]: rows.append({'configuration': LABEL[n], **{f'round {r_}': ms(np.array([RES[s][n]['curve'][r_ - 1] for s in SEEDS])) for r_ in rds}})
T['curves'] = pd.DataFrame(rows); print(T['curves'].to_string(index=False))

print('\n5e. PhysFed: do the PCR weights differ between clients? (mean over seeds and rounds)')
pf_ = RES[SEEDS[0]]['physfed_a05']
T['physfed_weights'] = pd.DataFrame([{'mean client PCR (gradient x input)': ms(G('physfed_a05', 'pcr_mean'), 4), 'std of PCR across clients': ms(G('physfed_a05', 'pcr_client_std'), 4), 'largest weight / uniform weight': ms(G('physfed_a05', 'pcr_weight_spread'), 3)}]); print(T['physfed_weights'].to_string(index=False))
print('   A largest-weight ratio near 1.0 means PhysFed is uniform weighting in disguise.')

print('\n5f. Differential privacy: client-level (epsilon, delta = %g), Renyi-DP composition over %d full-participation rounds, add/remove adjacency, clip C = %g' % (DP_DELTA, ROUNDS, DP_CLIP))
rows = [{'sigma': s_, 'epsilon': f'{RES[SEEDS[0]][f"dp_s{s_}"]["eps"]:.1f}', 'weighted F1': ms(G(f'dp_s{s_}', 'weighted'), 4), 'share of updates clipped': ms(G(f'dp_s{s_}', 'clip_frac')), 'benign recall': ms(np.array([RES[s][f"dp_s{s_}"]["recall_pc"][0] for s in SEEDS])), 'FedAvg (no noise)': ms(G('fedavg_a05', 'weighted'), 4)} for s_ in DP_SIGMAS]
T['dp'] = pd.DataFrame(rows); print(T['dp'].to_string(index=False))
print('   Only 9 clients are averaged: the noise vector has norm about sigma*C*sqrt(parameters)/9 (about 10*sigma for this model) against an averaged update of norm at most C, so a useful epsilon (below about 10) needs sigma near 3 or more. Epsilon values in the hundreds are not a meaningful guarantee.')

print('\n5g. White-box PGD (epsilon %.2f sigma, %d steps) against each final global model, test attack rows. Evasion includes rows already missed on clean data.' % (PGD_EPS, PGD_ITERS))
rows = []; pl = []
for n in names: pl.append(ptest(GA(n, 'evasion_pgd'), GA(n, 'evasion_clean')))
ph = holm(pl)
for i, n in enumerate(names): rows.append({'configuration': LABEL[n], 'clean F1': ms(G(n, 'weighted'), 3), 'F1 after PGD': ms(GA(n, 'f1_pgd'), 3), 'evasion clean': ms(GA(n, 'evasion_clean')), 'evasion PGD': ms(GA(n, 'evasion_pgd')), 'p (PGD vs clean evasion, Holm)': fp(ph[i])})
T['attack'] = pd.DataFrame(rows); print(T['attack'].to_string(index=False))
print('   A low PGD evasion on a model with low clean F1 is not robustness: compare the F1 columns.')

print('\n5h. Communication (per run, %d params x 4 bytes = %.1f KiB per update)' % (N_PARAMS, N_PARAMS * 4 / 1024))
up = N_PARAMS * 4 * N_CLIENTS * ROUNDS; both = 2 * up; raw = X_train.nbytes + y_train.nbytes
T['comm'] = pd.DataFrame([{'what': 'client -> server updates', 'MB': up / 1e6, 'share of raw data': up / raw}, {'what': 'updates + global model broadcast', 'MB': both / 1e6, 'share of raw data': both / raw}, {'what': 'raw training data (float32 features + labels)', 'MB': raw / 1e6, 'share of raw data': 1.0}]); print(T['comm'].to_string(index=False))
print('   "Less data transmitted" is a bandwidth statement, not a privacy statement; the only formal privacy bound here is the DP table above.')
T['partition'] = PART_TAB

## 6. Figures and saved outputs

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 11)); cmap = plt.get_cmap('tab10')
for i, n in enumerate(['fedavg_iid', 'fedavg_a05', 'fedprox_a05', 'fedavg_a01', 'fedprox_a01', 'fedavg_plc', 'physfed_a05', 'trimmed_a05', 'fedavg_byz_a05', 'trimmed_byz_a05']):
    c_ = np.stack([RES[s][n]['curve'] for s in SEEDS]); ax[0, 0].plot(range(1, ROUNDS + 1), c_.mean(0), label=n, color=cmap(i % 10), ls='-' if i < 7 else '--')
ax[0, 0].axhline(G('central_150', 'weighted').mean(), color='k', ls=':', label='central_150'); ax[0, 0].set_xlabel('round'); ax[0, 0].set_ylabel('test weighted F1'); ax[0, 0].set_title('Convergence (seed mean)'); ax[0, 0].legend(fontsize=7, ncol=2); ax[0, 0].grid(alpha=.3)
xs = np.arange(len(names)); ax[0, 1].bar(xs, [G(n, 'weighted').mean() for n in names], yerr=[G(n, 'weighted').std(ddof=1) for n in names], capsize=2); ax[0, 1].set_xticks(xs); ax[0, 1].set_xticklabels(names, rotation=75, fontsize=7); ax[0, 1].set_title('Final-round weighted F1'); ax[0, 1].grid(alpha=.3, axis='y')
ax[1, 0].bar(xs - 0.2, [GA(n, 'evasion_clean').mean() for n in names], 0.4, label='clean'); ax[1, 0].bar(xs + 0.2, [GA(n, 'evasion_pgd').mean() for n in names], 0.4, yerr=[GA(n, 'evasion_pgd').std(ddof=1) for n in names], capsize=2, label='after white-box PGD'); ax[1, 0].set_xticks(xs); ax[1, 0].set_xticklabels(names, rotation=75, fontsize=7); ax[1, 0].set_title('Evasion (share of attack rows labelled benign)'); ax[1, 0].legend(fontsize=8); ax[1, 0].grid(alpha=.3, axis='y')
eps = [RES[SEEDS[0]][f'dp_s{s_}']['eps'] for s_ in DP_SIGMAS]; ax[1, 1].errorbar(eps, [G(f'dp_s{s_}', 'weighted').mean() for s_ in DP_SIGMAS], yerr=[G(f'dp_s{s_}', 'weighted').std(ddof=1) for s_ in DP_SIGMAS], marker='o', capsize=3); ax[1, 1].axhline(G('fedavg_a05', 'weighted').mean(), color='k', ls=':', label='FedAvg, no noise'); ax[1, 1].set_xscale('log'); ax[1, 1].set_xlabel('epsilon (delta 1e-5, client-level)'); ax[1, 1].set_ylabel('weighted F1'); ax[1, 1].set_title('Privacy-utility trade-off'); ax[1, 1].legend(fontsize=8); ax[1, 1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'fl_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'tables': T, 'per_seed': {s: RES[s] for s in SEEDS}, 'config': {'N_CLIENTS': N_CLIENTS, 'ROUNDS': ROUNDS, 'LOCAL_EPOCHS': LOCAL_EPOCHS, 'BATCH': BATCH, 'LR': LR, 'TRIM': TRIM, 'BYZ_SCALE': BYZ_SCALE, 'DP_CLIP': DP_CLIP, 'DP_SIGMAS': DP_SIGMAS}}, f'fl_results_{PIPELINE_VERSION}.pkl')
for k_, t_ in T.items(): t_.to_csv(f'fl_{k_}_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB12', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'torch': torch.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb12_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'fl_results_{PIPELINE_VERSION}.pkl', f'fl_{PIPELINE_VERSION}.png', f'nb12_run_manifest_{PIPELINE_VERSION}.json'] + [f'fl_{k_}_{PIPELINE_VERSION}.csv' for k_ in T]: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)